# Fine-tune Laya for Low Gate

Teaches Laya to fly the Descent lander by copying the autopilot's pick from the 35-manoeuvre menu (asked as a tilt choice and a throttle score). Input: `train.jsonl` and `eval.jsonl` from `pnpm laya-dataset`, in `bridge/laya/data/<prompt>/`. Output: `<VERSION>.zip`, which goes back to the Mac and is converted to MLX there.

v1 and v2 were trained for the real-time arena (tag `v0-realtime`); v3 is the menu version.

| Version | Prompt | Data folder |
|---|---|---|
| laya-descent-v1 | p5 (full observation) | `bridge/laya/data/p5/` |
| laya-descent-v2 | p8 (brief observation, ~2x faster to answer) | `bridge/laya/data/p8/` |
| laya-descent-v3 | f3 (brief observation, menu: 7 tilts × 5 throttle levels; engine descent-3) | `bridge/laya/data/f3/` |
| laya-descent-v4 | f3, v3's data plus one DAgger round (states from v3's own flights) | `bridge/laya/data/f3-dagger1/` |

Before running: **Runtime → Change runtime type → A100 GPU** (L4 also works, slower). Then **Runtime → Run all**. The upload cell waits for you to pick the two files.

In [ ]:
VERSION = "laya-descent-v4"   # upload train.jsonl and eval.jsonl from bridge/laya/data/f3-dagger1/

In [ ]:
# 1. Check the GPU and install Laya (PyTorch is preinstalled on Colab).
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q laya==0.4.17
import importlib.metadata as md, torch
print("laya", md.version("laya"), "| torch", torch.__version__, "| CUDA", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> A100 GPU, then run again."

In [ ]:
# 2. Upload train.jsonl and eval.jsonl (click "Choose files", select both).
import json
from google.colab import files
uploaded = files.upload()
for name in ("train.jsonl", "eval.jsonl"):
    assert name in uploaded, f"{name} was not uploaded"
    print(name, sum(1 for _ in open(name)), "rows")
print("first state:\n" + json.loads(open("train.jsonl").readline())["state"])

In [ ]:
# 3. Download the base model (the PyTorch original of aac6fef/laya-mlx).
from huggingface_hub import snapshot_download
from laya.agent import _fix_tokenizer_config
snapshot_download("convaiinnovations/laya", local_dir="laya_base")
_fix_tokenizer_config("laya_base")

In [ ]:
# 4. Fine-tune. Full model (encoder and heads); 3 epochs over ~10.9k states.
#    eval.jsonl uses different courses, so the before/after numbers are held out.
import time
from laya.train import TrainConfig, finetune
config = TrainConfig(
    epochs=3,
    micro_batch=16,
    grad_accum=2,      # effective batch 32
    max_len=512,       # the English checkpoint's context; states are 100-300 tokens
    head_max_len=256,
    eval_data="eval.jsonl",
    seed=42,
)
started = time.time()
summary = finetune("train.jsonl", "laya_base", VERSION, config, device="cuda")
print(f"trained in {(time.time() - started) / 60:.1f} min")

In [ ]:
# 5. Before/after on held-out courses: accuracy should rise well above the base model's ~0.33.
report = json.load(open(f"{VERSION}/train_report.json"))
print("eval:", report["eval_mode"], report["eval_items"], "items")
for when in ("before", "after"):
    r = report[when]
    print(f"{when:6}  accuracy {r['accuracy']:.3f}  loss {r['loss']:.3f}  ece {r['ece']:.3f}")
print("temperatures", summary.get("temperature"))

In [ ]:
# 6. Zip the final model (without the per-epoch copy) and download it.
!cd {VERSION} && rm -rf checkpoint_latest && zip -qr ../{VERSION}.zip . && cd .. && ls -lh {VERSION}.zip
files.download(f"{VERSION}.zip")